# Modern Deep Learning Methods - Code

---

## Modeling Approach

In this notebook, we will import pre-trained ResNet-50 and EfficientNet-B4 models from PyTorch, replace the final classification layers for our problem, and perform cross-validation and test set evaluation as we did for our traditional CNN.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, TensorDataset
import copy
import random

In [2]:
# Set seeds for reproducibility

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
x_train = torch.load('data/x_train.pt')
y_train = torch.load('data/y_train.pt')
x_test = torch.load('data/x_test.pt')
y_test = torch.load('data/y_test.pt')

```{admonition} Note
As in our previous notebook, these code blocks were run on a GPU. If you cannot run these on a GPU, it's recommended you use smaller models, such as ResNet-18 or EfficientNet-B0, which are easier to run on CPU.
```

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## ResNet Definition

We will use a two-layer classifier head for our ResNet model, which includes two dropout layers (50% and 25%), and a final 5-neuron output layer for classification.

In [5]:
def create_resnet(num_classes=5, dropout_rate=0.5):
    model = models.resnet50(weights='DEFAULT')
    
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(dropout_rate/2),
        nn.Linear(512, num_classes)
    )
    return model

## EfficientNet Definition

For our EfficientNet model, we will use essentially the same classifier head, with slightly smaller initial dropout values due to the shallower architecture (deeper models are more prone to overfitting).

In [6]:
def create_efficientnet(num_classes=5, dropout_rate=0.3):
    model = models.efficientnet_b4(weights='DEFAULT')
    
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(512, num_classes)
    )
    return model

## Cost-Sensitive Learning

We trained our original CNN with cost-sensitive learning, and we will do the same for these models.

In [7]:
def get_class_weights(y_train, device):
    label_counts = Counter(y_train.tolist())
    num_classes = len(label_counts)
    total_samples = sum(label_counts.values())
    
    class_weights = [total_samples / (num_classes * label_counts[i]) for i in range(num_classes)]
    return torch.tensor(class_weights, dtype=torch.float).to(device)

## Model Training

We will use the same K-fold cross validation functions as before, but they will require different hyperparameters, which we will define below.

In [ ]:
def train_model(model_class, train_loader, val_loader, criterion, optimizer, epochs, patience):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    best_val_metrics = None
    model = model_class.to(device)
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                
                # Collect predictions and labels for metrics
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        
        # Early stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            
            # Calculate validation metrics for the best epoch
            val_labels_np = np.array(val_labels)
            val_preds_np = np.array(val_preds)
            
            qwk = cohen_kappa_score(val_labels_np, val_preds_np, weights='quadratic')
            precision, recall, f1, _ = precision_recall_fscore_support(
                val_labels_np, val_preds_np, average='weighted', zero_division=0
            )
            
            best_val_metrics = {
                'qwk': qwk,
                'precision': precision,
                'recall': recall,
                'f1': f1
            }
            
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs (no improvement for {patience} epochs)")
                break
    
    # Load best model state
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model with validation loss: {best_val_loss:.4f}")
    
    return model, best_val_metrics

:::{dropdown} Code Explanation
```python
def train_model(model_class, train_loader, val_loader, criterion, optimizer, epochs, patience):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    best_val_metrics = None
    model = model_class.to(device)
```
Initializes training by setting up early stopping variables, tracking the best validation loss and metrics, and moving the model to the appropriate device.

```python
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
```
Training phase: for each epoch, the model is trained on all batches in the training loader. Loss is accumulated to track average training performance.

```python
        model.eval()
        total_val_loss = 0
        val_preds = []
        val_labels = []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())

        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
```
Validation phase: the model is evaluated on the validation set without updating weights. It computes validation loss and collects predictions and true labels for metric computation.

```python
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())

            val_labels_np = np.array(val_labels)
            val_preds_np = np.array(val_preds)

            qwk = cohen_kappa_score(val_labels_np, val_preds_np, weights='quadratic')
            precision, recall, f1, _ = precision_recall_fscore_support(
                val_labels_np, val_preds_np, average='weighted', zero_division=0
            )

            best_val_metrics = {
                'qwk': qwk,
                'precision': precision,
                'recall': recall,
                'f1': f1
            }

            epochs_no_improve = 0
```
If the current validation loss is the best so far, save the model weights and compute evaluation metrics: quadratic weighted kappa, precision, recall, and F1-score.

```python
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs (no improvement for {patience} epochs)")
                break
```
Early stopping: if the validation loss hasn't improved for `patience` consecutive epochs, stop training early.

```python
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"Loaded best model with validation loss: {best_val_loss:.4f}")

    return model, best_val_metrics
```
After training, the best model weights are reloaded and both the model and the best validation metrics are returned.
:::

In [9]:
def cross_validate(model_class, X, y, epochs, patience, batch_size, lr, n_splits=5):
    print(f"Using device: {device}")
    print(f"Dataset: {len(X)} images")
    
    # Convert to numpy
    X = X.numpy()
    y = y.numpy()
    
    # Initialize cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    # Store results
    cv_results = {
        'qwk_scores': [],
        'precision_scores': [],
        'recall_scores': [],
        'f1_scores': []
    }
    
    # Cross-validation loop
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{n_splits} ---")
        
        # Split data
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Convert back to tensors
        X_train = torch.FloatTensor(X_train)
        X_val = torch.FloatTensor(X_val)
        y_train = torch.LongTensor(y_train)
        y_val = torch.LongTensor(y_val)
        
        # Create data loaders
        train_loader = DataLoader(TensorDataset(X_train, y_train), 
                                batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val, y_val), 
                               batch_size=batch_size, shuffle=False)

        # Initialize model
        model = model_class.to(device)
        
        # Calculate class weights for cost-sensitive learning
        class_weights = get_class_weights(y_train, device)
        print(f"Class weights: {class_weights.cpu().numpy()}")
        
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        
        # Train model with early stopping
        model, val_metrics = train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs, patience)
        
        # Use validation metrics from training
        qwk = val_metrics['qwk']
        precision = val_metrics['precision']
        recall = val_metrics['recall']
        f1 = val_metrics['f1']
        
        print(f"Fold {fold+1} Results:")
        print(f"  QWK: {qwk:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-score: {f1:.4f}")
        
        # Store results
        cv_results['qwk_scores'].append(qwk)
        cv_results['precision_scores'].append(precision)
        cv_results['recall_scores'].append(recall)
        cv_results['f1_scores'].append(f1)
    
    # Calculate and print summary statistics
    print(f"\n{'='*50}")
    print("CROSS-VALIDATION SUMMARY")
    print(f"{'='*50}")
    
    for metric, scores in cv_results.items():
        metric_name = metric.replace('_scores', '').replace('_', ' ').title()
        mean_score = np.mean(scores)
        print(f"{metric_name:12}: {mean_score:.4f}")
    
    return cv_results

:::{dropdown} Code Explanation
```python
def cross_validate(model_class, X, y, epochs, patience, batch_size, lr, n_splits=5):
    print(f"Using device: {device}")
    print(f"Dataset: {len(X)} images")
    
    X = X.numpy()
    y = y.numpy()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    cv_results = {
        'qwk_scores': [],
        'precision_scores': [],
        'recall_scores': [],
        'f1_scores': []
    }
```
Prepares for cross-validation by converting input tensors to NumPy arrays and initializing a `StratifiedKFold` splitter. An empty dictionary is created to collect metrics across folds.

```python
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Fold {fold+1}/{n_splits} ---")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        X_train = torch.FloatTensor(X_train)
        X_val = torch.FloatTensor(X_val)
        y_train = torch.LongTensor(y_train)
        y_val = torch.LongTensor(y_val)

        train_loader = DataLoader(TensorDataset(X_train, y_train), 
                                  batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val, y_val), 
                                batch_size=batch_size, shuffle=False)
```
Splits the data into training and validation sets for the current fold, converts them back into PyTorch tensors, and wraps them in DataLoaders for mini-batch training and evaluation.

```python
        model = model_class.to(device)

        class_weights = get_class_weights(y_train, device)
        print(f"Class weights: {class_weights.cpu().numpy()}")

        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        model, val_metrics = train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs, patience)
```
Initializes a fresh instance of the model, computes class weights to handle class imbalance, sets up the loss and optimizer, and trains the model using early stopping. Validation metrics are returned from `train_model`.

```python
        qwk = val_metrics['qwk']
        precision = val_metrics['precision']
        recall = val_metrics['recall']
        f1 = val_metrics['f1']

        print(f"Fold {fold+1} Results:")
        print(f"  QWK: {qwk:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-score: {f1:.4f}")

        cv_results['qwk_scores'].append(qwk)
        cv_results['precision_scores'].append(precision)
        cv_results['recall_scores'].append(recall)
        cv_results['f1_scores'].append(f1)
```
Extracts the validation metrics from the training result and logs them for this fold. The metrics are also saved to the `cv_results` dictionary for later summarization.

```python
    print(f"\n{'='*50}")
    print("CROSS-VALIDATION SUMMARY")
    print(f"{'='*50}")

    for metric, scores in cv_results.items():
        metric_name = metric.replace('_scores', '').replace('_', ' ').title()
        mean_score = np.mean(scores)
        print(f"{metric_name:12}: {mean_score:.4f}")

    return cv_results
```
After all folds are complete, computes and prints average scores for each metric across all folds. The results are returned for downstream use.
:::

## Cross Validation - ResNet-50

We will use a much smaller learning rate since this ResNet-50 model has been pretrained and will require small, careful adjustments to classify our images. These hyperparameters can be adjusted further.

In [ ]:
# ResNet Hyperparameters (Cross Validation)

resnet_cv_epochs = 20
resnet_cv_patience = 5
resnet_cv_batch_size = 32
resnet_cv_lr = 0.00001

resnet_class = create_resnet()
resnet_cv_results = cross_validate(resnet_class, 
                            x_train, 
                            y_train, 
                            resnet_cv_epochs, 
                            resnet_cv_patience, 
                            resnet_cv_batch_size, 
                            resnet_cv_lr)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 222MB/s]


Using device: cuda
Dataset: 2929 images

--- Fold 1/5 ---
Class weights: [0.40571427 1.9772152  0.73333335 3.809756   2.479365  ]
Epoch 1/20, Train Loss: 1.5888, Val Loss: 1.5569
Epoch 2/20, Train Loss: 1.5139, Val Loss: 1.4598
Epoch 3/20, Train Loss: 1.3948, Val Loss: 1.3296
Epoch 4/20, Train Loss: 1.2845, Val Loss: 1.2528
Epoch 5/20, Train Loss: 1.1912, Val Loss: 1.1856
Epoch 6/20, Train Loss: 1.0995, Val Loss: 1.1254
Epoch 7/20, Train Loss: 1.0077, Val Loss: 1.0612
Epoch 8/20, Train Loss: 0.9008, Val Loss: 1.0294
Epoch 9/20, Train Loss: 0.8080, Val Loss: 1.0073
Epoch 10/20, Train Loss: 0.6984, Val Loss: 0.9912
Epoch 11/20, Train Loss: 0.6174, Val Loss: 0.9895
Epoch 12/20, Train Loss: 0.5248, Val Loss: 1.0382
Epoch 13/20, Train Loss: 0.4421, Val Loss: 1.0585
Epoch 14/20, Train Loss: 0.3662, Val Loss: 1.1129
Epoch 15/20, Train Loss: 0.2830, Val Loss: 1.1410
Epoch 16/20, Train Loss: 0.2451, Val Loss: 1.1963
Early stopping triggered after 16 epochs (no improvement for 5 epochs)
Loaded b

## Cross Validation - EfficientNet-B4

Parameters similar to those we used for our ResNet model work well with EfficientNet too. Again, try to adjust epochs, batch_size, and learning rate to see how they affect results.

In [ ]:
# EfficientNet Hyperparameters (Cross Validation)

effnet_cv_epochs = 30
effnet_cv_patience = 5
effnet_cv_batch_size = 32
effnet_cv_lr = 0.00005

effnet_class = create_efficientnet()
effnet_cv_results = cross_validate(effnet_class, 
                                   x_train, 
                                   y_train, 
                                   effnet_cv_epochs,
                                   effnet_cv_patience,
                                   effnet_cv_batch_size,
                                   effnet_cv_lr)

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:03<00:00, 19.8MB/s]


Using device: cuda
Dataset: 2929 images

--- Fold 1/5 ---
Class weights: [0.40571427 1.9772152  0.73333335 3.809756   2.479365  ]
Epoch 1/30, Train Loss: 1.5499, Val Loss: 1.4398
Epoch 2/30, Train Loss: 1.3345, Val Loss: 1.2278
Epoch 3/30, Train Loss: 1.1436, Val Loss: 1.0714
Epoch 4/30, Train Loss: 0.9600, Val Loss: 0.9916
Epoch 5/30, Train Loss: 0.8753, Val Loss: 0.9774
Epoch 6/30, Train Loss: 0.8108, Val Loss: 0.9775
Epoch 7/30, Train Loss: 0.7233, Val Loss: 0.9655
Epoch 8/30, Train Loss: 0.6929, Val Loss: 0.9586
Epoch 9/30, Train Loss: 0.6156, Val Loss: 0.9586
Epoch 10/30, Train Loss: 0.5552, Val Loss: 0.9441
Epoch 11/30, Train Loss: 0.5254, Val Loss: 1.0291
Epoch 12/30, Train Loss: 0.4404, Val Loss: 1.0805
Epoch 13/30, Train Loss: 0.4005, Val Loss: 1.0481
Epoch 14/30, Train Loss: 0.3868, Val Loss: 1.1599
Epoch 15/30, Train Loss: 0.3558, Val Loss: 1.1499
Early stopping triggered after 15 epochs (no improvement for 5 epochs)
Loaded best model with validation loss: 0.9441
Fold 1 Resu

In [12]:
def evaluate_final_model(model_class, X_train, y_train, X_test, y_test, epochs, batch_size, lr):        
    train_loader = DataLoader(TensorDataset(X_train, y_train), 
                            batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), 
                           batch_size=batch_size, shuffle=False)

    # Initialize model
    model = model_class.to(device)
    
    # Setup training
    class_weights = get_class_weights(y_train, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Train on entire train set (no early stopping)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
    
    # Evaluate on test set
    print(f"\nEvaluating on test set ({len(X_test)} samples)...")
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    
    # Calculate metrics
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )
    
    print(f"\nFinal Test Results:")
    print(f"  QWK: {qwk:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    
    return model, y_true, y_pred

:::{dropdown} Code Explanation
```python
def evaluate_final_model(model_class, X_train, y_train, X_test, y_test, epochs, batch_size, lr):        
    train_loader = DataLoader(TensorDataset(X_train, y_train), 
                            batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), 
                           batch_size=batch_size, shuffle=False)
```
Wraps the full training and test datasets in PyTorch `DataLoader` objects for batched training and evaluation. Training data is shuffled; test data is not.

```python
    model = model_class.to(device)
    
    class_weights = get_class_weights(y_train, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
```
Initializes the model and training components: class-weighted loss function (for imbalanced classes), and the Adam optimizer with the specified learning rate.

```python
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
```
Trains the model for the specified number of epochs on the full training set. Unlike earlier training loops, this does not include early stopping or validation — it's a final fit.

```python
    print(f"\nEvaluating on test set ({len(X_test)} samples)...")
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
```
Evaluates the trained model on the test set. Predictions and true labels are collected across all batches and converted into NumPy arrays for metric calculation.

```python
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    print(f"\nFinal Test Results:")
    print(f"  QWK: {qwk:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    return model, y_true, y_pred
```
Computes final evaluation metrics: Quadratic Weighted Kappa, Precision, Recall, and F1-score. These are printed and returned along with the model and test predictions.
:::

## Evaluation on Test Set

Based on our cross validation results, we will use only a few epochs and keep very low learning rates.

In [ ]:
# ResNet Hyperparameters (Test)

resnet_epochs = 4
resnet_batch_size = 32
resnet_lr = 0.00001

resnet_model, y_true, y_pred = evaluate_final_model(resnet_class, 
                                                    x_train, 
                                                    y_train, 
                                                    x_test, 
                                                    y_test,
                                                    resnet_epochs,
                                                    resnet_batch_size,
                                                    resnet_lr)

Epoch 1/4, Loss: 0.1945
Epoch 2/4, Loss: 0.1616
Epoch 3/4, Loss: 0.1443
Epoch 4/4, Loss: 0.1342

Evaluating on test set (733 samples)...

Final Test Results:
  QWK: 0.8136, Precision: 0.7582, Recall: 0.7653, F1: 0.7606


In [ ]:
# EfficientNet Hyperparameters (Test)

effnet_epochs = 5
effnet_batch_size = 32
effnet_lr = 0.00005

effnet_model, y_true, y_pred = evaluate_final_model(effnet_class, 
                                                    x_train, 
                                                    y_train, 
                                                    x_test, 
                                                    y_test,
                                                    effnet_epochs,
                                                    effnet_batch_size,
                                                    effnet_lr)

Epoch 1/5, Loss: 0.3009
Epoch 2/5, Loss: 0.2647
Epoch 3/5, Loss: 0.2274
Epoch 4/5, Loss: 0.2598
Epoch 5/5, Loss: 0.2275

Evaluating on test set (733 samples)...

Final Test Results:
  QWK: 0.8339, Precision: 0.7956, Recall: 0.7749, F1: 0.7796


In both models, we see significant improvements across all metrics from our baseline CNN. EfficientNet-B4 performs consistently better than ResNet-50. In the next section, we'll dive deeper into the results and look at the class-by-class performance of both models.

In [ ]:
# Save models for results section

torch.save(resnet_model.state_dict(), 'models/resnet_state_dict.pth')
torch.save(effnet_model.state_dict(), 'models/efficientnet_state_dict.pth')